## Youtube-Video-Downloder
## video link "https://www.youtube.com/watch?v=TZwdH14f1fI"

In [ ]:
import yt_dlp
import os

# Enter your YouTube Video Link here
youtube_url = "https://www.youtube.com/watch?v=TZwdH14f1fI"
# Path where the output video will be saved
output_video_path = '../videos/1080Video.mp4'

print("Downloading 1080p video from YouTube... Please wait a moment ⏳")

# Create a folder to save videos if it doesn't exist
os.makedirs(os.path.dirname(output_video_path), exist_ok=True)

# yt-dlp settings (For a maximum resolution of 1080p)
ydl_opts = {
    # [height<=1080] ensures that the video quality does not exceed 1080p
    'format': 'bestvideo[height<=1080][ext=mp4]+bestaudio[ext=m4a]/best[height<=1080][ext=mp4]/best',
    'outtmpl': output_video_path,
    'quiet': False, # Setting this to True will hide the download logs
    'noplaylist': True,
    'merge_output_format': 'mp4' # This ensures that the final file is in mp4 format
}

# First check if an old file exists, and delete it if so
if os.path.exists(output_video_path):
    os.remove(output_video_path)

# Video Download
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])
    print(f"\nVideo successfully downloaded in 1080p! \nLocation: {output_video_path}")
except Exception as e:
    print(f"\nAn error occurred during download: {e}")

YouTube se 1080p video download ho raha hai... Thoda wait karein ⏳
[youtube] Extracting URL: https://www.youtube.com/watch?v=TZwdH14f1fI
[youtube] TZwdH14f1fI: Downloading webpage


[youtube] TZwdH14f1fI: Downloading android vr player API JSON
[info] TZwdH14f1fI: Downloading 1 format(s): 399+140
[download] Destination: ..\videos\1080Video.f399.mp4
[download] 100% of  129.78MiB in 00:00:11 at 11.64MiB/s    
[download] Destination: ..\videos\1080Video.f140.m4a
[download] 100% of    8.51MiB in 00:00:00 at 10.90MiB/s  
[Merger] Merging formats into "..\videos\1080Video.mp4"
Deleting original file ..\videos\1080Video.f399.mp4 (pass -k to keep)
Deleting original file ..\videos\1080Video.f140.m4a (pass -k to keep)

Video successfully 1080p mein download ho gaya! 
Location: ../videos/1080Video.mp4


# traker

## 1. Imports and Dependencies
Importing all the necessary libraries required for video processing, mathematical operations, clustering (KMeans), and object detection/tracking (YOLO).
*(Note: Ensure you have `opencv-python`, `numpy`, `scikit-learn`, and `ultralytics` installed in your `UV` environment).*

In [9]:
import cv2
import math
import os
import numpy as np 
from sklearn.cluster import KMeans
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator

## 2. Configuration & Setup
Defining the file paths for the model, input video, and output directory. This cell also sets up the YOLO class IDs, tracking heuristics (to fix ball tracking lag), and the team colors in BGR format. A cache dictionary is initialized to store player team colors and optimize performance.

In [14]:
# CONFIGURATION & SETUP

MODEL_PATH = '../model/run_1/weights/best.pt'
INPUT_VIDEO = '../videos/1080Video.mp4'

OUTPUT_DIR = '../process_videos/match_highlights'
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, 'output.mp4')

# YOLO Custom Model Classes
BALL_CLASS_ID = 0    
PLAYER_CLASS_ID = 2  

# Ball Tracking Heuristics (To fix lag)
MAX_PIXEL_JUMP = 250  # Allowed distance for fast kicks
LOST_PATIENCE = 15    # How many frames to wait if the ball disappears

# --- Team Colors (BGR Format) ---
TEAM_RED_BGR = np.array([0, 0, 255])       # Red Team (Blue=0, Green=0, Red=255)
TEAM_SKY_BLUE_BGR  = np.array([235, 206, 135]) # Sky Blue Team

# Cache to remember Player IDs and their respective teams
player_team_cache = {}

## 3. Utility Functions
Helper functions to calculate the center coordinates of bounding boxes, measure the Euclidean distance between two points, and draw a neat triangle marker directly above the ball.

In [15]:
# UTILITY FUNCTIONS

def get_center(coords):
    """Calculate the center point of a bounding box."""
    x1, y1, x2, y2 = coords
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def distance(p1, p2):
    """Calculate the distance between two points."""
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def draw_triangle(frame, bbox, color):
    """Draw a neat triangle marker above the ball."""
    y = int(bbox[1])
    x = int((bbox[0] + bbox[2]) / 2) 
    triangle_points = np.array([[x, y], [x-12, y-22], [x+12, y-22]])
    
    cv2.drawContours(frame, [triangle_points], 0, color, cv2.FILLED)
    cv2.drawContours(frame, [triangle_points], 0, (0, 0, 0), 2) # Black border
    return frame

## 4. Team Classification Functions
Functions dedicated to distinguishing between teams based on jersey colors. It converts colors to LAB space for accurate comparison, isolates the player's upper body, masks out the green grass background, and uses KMeans clustering to find the dominant color of the jersey. Finally, it matches the dominant color to either the Red or Sky Blue team.

In [16]:
# TEAM CLASSIFICATION FUNCTIONS

def get_color_distance_lab(bgr1, bgr2):
    """Calculate accurate difference between two colors in LAB color space."""
    c1, c2 = np.uint8([[bgr1]]), np.uint8([[bgr2]])
    lab1 = cv2.cvtColor(c1, cv2.COLOR_BGR2LAB)[0][0].astype(float)
    lab2 = cv2.cvtColor(c2, cv2.COLOR_BGR2LAB)[0][0].astype(float)
    return np.linalg.norm(lab1 - lab2)

def get_dominant_color(player_crop):
    """Extract the main color of the player's shirt (ignoring grass)."""
    height, width, _ = player_crop.shape
    
    # Crop only the upper body (jersey) area
    y1, y2 = int(height * 0.15), int(height * 0.45) 
    x1, x2 = int(width * 0.25), int(width * 0.75)
    jersey_crop = player_crop[y1:y2, x1:x2]
    
    if jersey_crop.size == 0: return None

    # Remove grass (Green) color
    hsv_crop = cv2.cvtColor(jersey_crop, cv2.COLOR_BGR2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv_crop, lower_green, upper_green)
    
    pixels = jersey_crop.reshape(-1, 3)
    valid_pixels = pixels[mask.flatten() == 0] # Pixels that are not green
    
    if len(valid_pixels) < 10: return None
    
    # Use KMeans to extract the dominant color
    kmeans = KMeans(n_clusters=1, n_init='auto').fit(valid_pixels)
    return kmeans.cluster_centers_[0]

def get_team_and_color(dom_color_bgr):
    """Match the dominant color to a specific team (Red vs Sky Blue)."""
    d_red = get_color_distance_lab(dom_color_bgr, TEAM_RED_BGR)
    d_sky = get_color_distance_lab(dom_color_bgr, TEAM_SKY_BLUE_BGR)
    
    # Assign Team and return Circle/Text color (BGR format)
    if d_red < d_sky:
        return "T-Red", (0, 0, 255)   
    else:
        return "T-Sky", (235, 206, 135)

## 5. Initialization & Video Setup
Loading the YOLO model weights and opening the input video stream. This cell also extracts video properties (width, height, fps) to initialize the `cv2.VideoWriter` for saving the output, alongside setting up tracking states for the ball.

In [17]:
# INITIALIZATION & VIDEO SETUP

print("Loading YOLO model...")
model = YOLO(MODEL_PATH)

print(f"Opening video: {INPUT_VIDEO}")
cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    print(f"ERROR: Video file not found! Please check the path: {INPUT_VIDEO}")
    # In Jupyter, raising an exception stops the execution cleanly
    raise FileNotFoundError(f"Video not found at {INPUT_VIDEO}")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# Ball State Variables
last_ball_pos = None
frames_since_last_ball = 0
frame_count = 0

Loading YOLO model...
Opening video: ../videos/1080Video.mp4


## 6. Main Processing Loop
The core loop of the pipeline. It reads the video frame-by-frame, runs YOLO object tracking, and iterates through detections. For players, it checks the team color cache or calculates the dominant color to draw 3D-like ellipses at their feet. For the ball, it uses tracking heuristics to filter out false positives and draws the triangle marker, before writing the annotated frame to the output video.

In [18]:
# MAIN PROCESSING LOOP

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: 
        break 
    
    frame_count += 1
    if frame_count % 30 == 0: 
        print(f"Processing frame: {frame_count} / {total_frames}")

    # YOLO Tracking (Confidence 0.25 to catch blurry ball movements)
    results = model.track(frame, conf=0.25, iou=0.5, persist=True, verbose=False)
    result = results[0]

    annotator = Annotator(frame)
    ball_candidates = []
    
    if result.boxes is not None:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            coords = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])

            # --- A. PLAYER LOGIC ---
            if cls_id == PLAYER_CLASS_ID:
                track_id = int(box.id[0]) if box.id is not None else -1
                team_name, box_color = "Unknown", (255, 255, 255)
                x1, y1, x2, y2 = coords.astype(int)

                if track_id != -1:
                    if track_id in player_team_cache:
                        # Fetch color from cache (Fast performance)
                        team_name, box_color = player_team_cache[track_id]
                    else:
                        # New player found, calculate color
                        # Ensure crop limits are inside the frame boundaries
                        crop_y1, crop_y2 = max(0, y1), min(height, y2)
                        crop_x1, crop_x2 = max(0, x1), min(width, x2)
                        
                        player_crop = frame[crop_y1:crop_y2, crop_x1:crop_x2]
                        dom_color = get_dominant_color(player_crop)
                        
                        if dom_color is not None:
                            team_name, box_color = get_team_and_color(dom_color)
                            player_team_cache[track_id] = (team_name, box_color)

                # --- DRAWING LOGIC (Only Circle & Small Text) ---
                if track_id != -1 and team_name != "Unknown":
                    label = f"{team_name} id {track_id}"
                else:
                    label = ""

                center_x = int((x1 + x2) / 2)
                bottom_y = int(y2)
                radius = int((x2 - x1) / 2) 

                # 1. 3D-like circle (ellipse) at player's feet
                cv2.ellipse(frame, (center_x, bottom_y), (radius, int(radius * 0.3)), 0, 0, 360, box_color, 2)

                # 2. Small text beneath the circle
                if label:
                    text_x = center_x - 30 
                    text_y = bottom_y + 15
                    
                    # Black outline first, then actual color for text visibility
                    cv2.putText(frame, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2, cv2.LINE_AA)
                    cv2.putText(frame, label, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.4, box_color, 1, cv2.LINE_AA)

            # --- B. BALL CANDIDATE COLLECTION ---
            elif cls_id == BALL_CLASS_ID:
                ball_candidates.append(box)

    # --- C. SMART BALL TRACKING HEURISTICS ---
    best_ball_box = None
    
    if ball_candidates:
        if last_ball_pos is None:
            # First time or after a gap: Pick the ball with the highest confidence
            best_ball_box = max(ball_candidates, key=lambda b: float(b.conf[0]))
        else:
            # Find the nearest ball from the last known position
            closest_ball = None
            min_dist = float('inf')
            
            for candidate in ball_candidates:
                cand_center = get_center(candidate.xyxy[0].cpu().numpy())
                d = distance(cand_center, last_ball_pos)
                if d < min_dist:
                    min_dist = d
                    closest_ball = candidate
            
            # If the ball hasn't jumped beyond the allowed pixel limit in one frame
            if min_dist <= MAX_PIXEL_JUMP:
                best_ball_box = closest_ball

    # --- D. DRAWING BALL MARKER ---
    annotated_frame = annotator.result()
    
    if best_ball_box is not None:
        last_ball_pos = get_center(best_ball_box.xyxy[0].cpu().numpy())
        frames_since_last_ball = 0
        annotated_frame = draw_triangle(annotated_frame, best_ball_box.xyxy[0].cpu().numpy(), (0, 255, 0))
    else:
        frames_since_last_ball += 1
        # If the ball has been lost for too long, clear memory
        if frames_since_last_ball > LOST_PATIENCE:
            last_ball_pos = None 

    # Save output frame
    out.write(annotated_frame)

Processing frame: 30 / 13783
Processing frame: 60 / 13783
Processing frame: 90 / 13783
Processing frame: 120 / 13783
Processing frame: 150 / 13783
Processing frame: 180 / 13783
Processing frame: 210 / 13783
Processing frame: 240 / 13783
Processing frame: 270 / 13783
Processing frame: 300 / 13783
Processing frame: 330 / 13783
Processing frame: 360 / 13783
Processing frame: 390 / 13783
Processing frame: 420 / 13783
Processing frame: 450 / 13783
Processing frame: 480 / 13783
Processing frame: 510 / 13783
Processing frame: 540 / 13783
Processing frame: 570 / 13783
Processing frame: 600 / 13783
Processing frame: 630 / 13783
Processing frame: 660 / 13783
Processing frame: 690 / 13783
Processing frame: 720 / 13783
Processing frame: 750 / 13783
Processing frame: 780 / 13783
Processing frame: 810 / 13783
Processing frame: 840 / 13783
Processing frame: 870 / 13783
Processing frame: 900 / 13783
Processing frame: 930 / 13783
Processing frame: 960 / 13783
Processing frame: 990 / 13783
Processing fr

## 7. Cleanup
Releases the video capture and writer objects to free up system resources and finalize the output video file.

In [20]:
cap.release()
out.release()
print(f"Video Processing Complete! Output saved at: {OUTPUT_VIDEO}")

Video Processing Complete! Output saved at: ../process_videos/match_highlights\output.mp4
